In [ ]:
!pip install crewai -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 12.5 MB/s e

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "Your_API_KEY"

In [ ]:
from crewai.flow.flow import Flow, listen, start
from openai import OpenAI

In [ ]:
from crewai.flow.flow import Flow, listen, start
from litellm import completion


class DishFlow(Flow):
    model = "gpt-4o-mini"

    @start()
    def suggest_cuisine(self):
        response = completion(
            model=self.model,
            messages=[
                {"role": "user", "content": "Suggest a random world cuisine."}
            ],
        )
        cuisine = response["choices"][0]["message"]["content"].strip()
        self.state["cuisine"] = cuisine
        return cuisine

    @listen(suggest_cuisine)
    def suggest_dish(self, cuisine):
        response = completion(
            model=self.model,
            messages=[
                {"role": "user", "content": f"Name one famous dish from {cuisine} cuisine."}
            ],
        )
        dish = response["choices"][0]["message"]["content"].strip()
        self.state["dish"] = dish
        return dish  # only return the dish

    @listen(suggest_dish)
    def give_tip(self, dish):
        response = completion(
            model=self.model,
            messages=[
                {"role": "user", "content": f"Give a useful cooking tip for making {dish}."}
            ],
        )
        tip = response["choices"][0]["message"]["content"].strip()
        self.state["tip"] = tip
        return tip   # only return the tip

    @listen(give_tip)
    def summarize(self, tip):
        cuisine = self.state.get("cuisine", "Unknown cuisine")
        dish = self.state.get("dish", "Unknown dish")

        summary = (
            f"Cuisine: {cuisine}\n"
            f"Dish: {dish}\n"
            f"Tip: {tip}"
        )
        self.state["summary"] = summary
        return summary

In [ ]:
flow = DishFlow()
flow.plot("dishflow.html")
result = flow.kickoff()

print("\n=== Final Output ===")
print(result)

╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: DishFlow                                                                                                 │
│  ID: 4feb04a3-4c38-49bd-b58f-9757f6e8d43e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 4feb04a3-4c38-49bd-b58f-9757f6e8d43e

Output()

Plot saved as dishflow.html.html


╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: DishFlow                                                                                                 │
│  ID: 4feb04a3-4c38-49bd-b58f-9757f6e8d43e                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== Final Output ===
Cuisine: How about exploring Ethiopian cuisine? It's known for its unique flavors, spices, and communal eating style. Dishes are often served on injera, a sourdough flatbread made from teff flour, which doubles as both a plate and an accompaniment. Popular dishes include doro wat (spicy chicken stew) and various lentil and vegetable stews called wats. Don't forget the traditional practice of sharing food and eating with your hands!
Dish: One famous dish from Ethiopian cuisine is **doro wat**, which is a spicy chicken stew simmered with berbere spice, onions, garlic, and ginger. It is often served with injera, and it is a staple during special occasions and holidays. The dish is typically enjoyed in a communal setting, where diners share from a common plate using pieces of injera to scoop up the stews.
Tip: A useful cooking tip for making **doro wat** is to allow the onions to cook down thoroughly before adding the spices and chicken. Traditional recipes often call